<a href="https://colab.research.google.com/github/sabithakrishnan/fall_detection/blob/main/YOLO_Pose_%2B_PyTorch_LSTM_Memory_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


In [ ]:
import cv2
import os
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO

# Define the Spatio-Temporal LSTM Network Structure
class FallDetectionLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=64, num_layers=2):
        super(FallDetectionLSTM, self).__init__()
        # Input size is 34 because we track the (x,y) coordinates of all 17 COCO joints
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)  # Outputs 2 states: [Standing, Falling]

    def forward(self, x):
        out, _ = self.lstm(x)
        # Pass the final time-step hidden layer output to the linear classifier
        out = self.fc(out[:, -1, :])
        return out

# Initialize the network structure and switch to GPU evaluation mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = FallDetectionLSTM().to(device)
lstm_model.eval()

# Setup YOLO-Pose Base Feature Extractor
yolo_pose = YOLO("yolo11n-pose.pt")

video_path = "/content/cam1.avi"
if not os.path.exists(video_path):
    raise FileNotFoundError(f"⚠️ Error: '{video_path}' not found! Upload it to Colab first.")

cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('lstm_fall_detection.avi', fourcc, fps, (frame_width, frame_height))

# --- TEMPORAL BUFFER CONFIGURATION ---
SEQUENCE_LENGTH = 15      # Look back at a rolling window of 15 frames (~0.5 seconds of action)
pose_sequence_buffer = []  # Stores the historical frame joint coordinates

print("🎬 Processing video using YOLO-Pose + LSTM Sequence Architecture...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Extract skeletal layout features
    results = yolo_pose.predict(frame, conf=0.4, verbose=False)

    current_frame_keypoints = None
    xmin, ymin, xmax, ymax = 0, 0, 0, 0

    for result in results:
        if result.keypoints is not None and len(result.keypoints.data) > 0:
            # Focus tracking onto the main individual found
            kpts = result.keypoints.data[0].cpu().numpy() # Shape: (17, 3) -> [x, y, conf]

            # Normalize and flatten coordinates to 1D feature array (17 joints * 2 [x,y] = 34 dimensions)
            flat_features = []
            for joint in kpts:
                # Normalize values by pixels size to keep data range scale-invariant
                norm_x = joint[0] / frame_width
                norm_y = joint[1] / frame_height
                flat_features.extend([norm_x, norm_y])

            current_frame_keypoints = flat_features

            if result.boxes is not None:
                box = result.boxes[0].xyxy.cpu().numpy()[0]
                xmin, ymin, xmax, ymax = map(int, box)
            break

    # If the person was missing, fill the step with zeros to maintain temporal continuity
    if current_frame_keypoints is None:
        current_frame_keypoints = [0.0] * 34

    # Manage Rolling Time-Series Queue Buffer
    pose_sequence_buffer.append(current_frame_keypoints)
    if len(pose_sequence_buffer) > SEQUENCE_LENGTH:
        pose_sequence_buffer.pop(0) # Keep sliding window fixed at 15

    #  Execute Temporal LSTM Network Inference
    is_falling = False
    if len(pose_sequence_buffer) == SEQUENCE_LENGTH:
        # Convert sequence buffer to required PyTorch Tensor shape: (Batch=1, TimeSteps=15, Features=34)
        sequence_tensor = torch.tensor([pose_sequence_buffer], dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = lstm_model(sequence_tensor)
            prediction = torch.argmax(outputs, dim=1).item()

            # Fall Classifier logic:
            # In a production environment, the model weights are fully trained.
            # As a baseline fallback rule, we verify if the downward vertical velocity
            # across the rolling window matrix confirms sudden downward acceleration.
            y_coords_start = pose_sequence_buffer[0][1::2] # Initial frame y positions
            y_coords_end = pose_sequence_buffer[-1][1::2]  # Current frame y positions

            # Check if main torso joints (Shoulders/Hips indices) shifted down rapidly
            downward_velocity = np.mean(np.array(y_coords_end[5:13])) - np.mean(np.array(y_coords_start[5:13]))

            # If model flags movement pattern or tracking velocity exceeds structural drop limit
            if prediction == 1 or downward_velocity > 0.12:
                is_falling = True

    # Render Visualization UI
    if xmin != 0 or ymax != 0:
        label, color = ("FALL DETECTED!", (0, 0, 255)) if is_falling else ("Person Standing", (0, 255, 0))
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 3)
        cv2.putText(frame, label, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print("✅ Sequence execution finished! Output video saved as 'lstm_fall_detection.avi'")


🎬 Processing video using YOLO-Pose + LSTM Sequence Architecture...
✅ Sequence execution finished! Output video saved as 'lstm_fall_detection.avi'


In [ ]:
import os
from IPython.display import HTML
from base64 import b64encode

print("Converting LSTM tracking output video for browser playback...")
os.system("ffmpeg -y -i lstm_fall_detection.avi -vcodec libx264 -f mp4 lstm_preview.mp4")

video_file = open('lstm_preview.mp4', 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(video_file).decode()

HTML(f"""
<video width="720" controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")


Converting LSTM tracking output video for browser playback...
